### 1 . Load 'coaching_marks_clean.csv' in a notebook named 'coaching_quality.ipynb' and build 'coaching_dirty' with 162 rows: use `iloc` with `list(range(150)) + list(range(12))` to repeat the first 12 students, then use `loc` to set fees_paid_inr to 250000 for the labels 20, 55, 90 and 130. Hint: if the file is missing, generate 150 rows with the same columns using `np.random.seed(42)`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('coaching_marks_clean.csv')

coaching_dirty = df.iloc[list(range(150)) + list(range(12))].copy()

coaching_dirty.loc[[20, 55, 90, 130], 'fees_paid_inr'] = 250000

print("Shape:", coaching_dirty.shape)
print(coaching_dirty.head())
print(coaching_dirty.loc[[20, 55, 90, 130]])

Shape: (162, 7)
   student_id batch  maths  science  english  attendance_pct  fees_paid_inr
0           1     C   38.0     66.0       46            86.0        19621.0
1           2     A   57.0     67.0       89            59.0        30034.0
2           3     C   49.0     52.0       47            59.0        28623.0
3           4     C   77.0     59.0       57            68.0        20126.0
4           5     A   63.0     88.0       64            66.0        20122.0
     student_id batch  maths  science  english  attendance_pct  fees_paid_inr
20           21     A   96.0     76.0       86            80.0       250000.0
55           56     C   89.0     87.0       70            83.0       250000.0
90           91     A   43.0     91.0       52            53.0       250000.0
130         131     C   62.0     36.0       67            97.0       250000.0


### 

### 2 .Find the repeated rows of 'coaching_dirty' with `duplicated()` and print their count, which must be 12. Then create 'coaching_nodup' with `drop_duplicates()` and check in code that it has 150 rows.

In [3]:
duplicates = coaching_dirty.duplicated()

print("Duplicate rows:", duplicates.sum())

coaching_nodup = coaching_dirty.drop_duplicates()

print("Rows after removing duplicates:", len(coaching_nodup))

assert duplicates.sum() == 12
assert len(coaching_nodup) == 150

print("Both checks passed!")


Duplicate rows: 12
Rows after removing duplicates: 150
Both checks passed!


### 3.Detect outliers in fees_paid_inr of 'coaching_nodup' with the IQR method: get Q1 and Q3 with `quantile`, set the fences at 1.5 x IQR and store the flagged rows in 'fee_outliers_iqr'. Print their student_id values; the 4 planted students must be among them. Hint: `df[(df['col'] < low) | (df['col'] > high)]`.

In [ ]:

Q1 = coaching_nodup['fees_paid_inr'].quantile(0.25)
Q3 = coaching_nodup['fees_paid_inr'].quantile(0.75)

IQR = Q3 - Q1

low = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR

fee_outliers_iqr = coaching_nodup[
    (coaching_nodup['fees_paid_inr'] < low) |                                          # Detect outliers      
    (coaching_nodup['fees_paid_inr'] > high)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Fence:", low)
print("Upper Fence:", high)

print("Outlier Student IDs:")
print(fee_outliers_iqr['student_id'].tolist())

print("Total Outliers:", len(fee_outliers_iqr))


Q1: 21333.25
Q3: 31637.0
IQR: 10303.75
Lower Fence: 5877.625
Upper Fence: 47092.625
Outlier Student IDs:
[21, 56, 91, 131]
Total Outliers: 4


### 4.Add a z-score column 'fee_z' for fees_paid_inr using (value - mean) / std from pandas `mean()` and `std()`, store the rows with |z| above 3 in 'fee_outliers_z' and print how many rows each method flagged. Finish by saving 'coaching_nodup' without the IQR outliers as 'coaching_final.csv' with `index=False`. Hint: pandas `std()` uses the sample formula (ddof=1) by default.

In [5]:
mean_fee = coaching_nodup['fees_paid_inr'].mean()
std_fee = coaching_nodup['fees_paid_inr'].std()                                 # Calculate mean and standard deviation

coaching_nodup['fee_z'] = (                                                             # Calculate Z-score
    coaching_nodup['fees_paid_inr'] - mean_fee
) / std_fee

fee_outliers_z = coaching_nodup[
    coaching_nodup['fee_z'].abs() > 3                                                 # Detect Z-score outliers
]

print("IQR Outliers:", len(fee_outliers_iqr))
print("Z-score Outliers:", len(fee_outliers_z))

coaching_final = coaching_nodup.drop(fee_outliers_iqr.index)                             # Remove IQR outliers

coaching_final.to_csv('coaching_final.csv', index=False)

print("Final rows:", len(coaching_final))
print("File saved successfully!")

IQR Outliers: 4
Z-score Outliers: 4
Final rows: 146
File saved successfully!
